# Graph Feature Construction

Converts formatted transaction CSV into PyTorch Geometric graph tensors.


## 1. Load Formatted Transactions


In [13]:
import os
import pandas as pd
import numpy as np
import torch

csv_path = "formatted_transactions.csv"
if not os.path.exists(csv_path):
    csv_path = os.path.join("model", "formatted_transactions.csv")

df_edges = pd.read_csv(csv_path)
print(f"Loaded {len(df_edges):,} transactions from {csv_path}")


Loaded 5,000,000 transactions from formatted_transactions.csv
Columns : ['EdgeID', 'from_id', 'to_id', 'Timestamp', 'Amount Sent', 'Sent Currency', 'Amount Received', 'Received Currency', 'Payment Format', 'Is Laundering']


,EdgeID,from_id,to_id,Timestamp,Amount Sent,Sent Currency,Amount Received,Received Currency,Payment Format,Is Laundering
0,1253484,920505,935525,19810,2825.70,10,2825.70,10,3,0
1,1284010,958125,1151467,19810,272.87,14,272.87,14,2,0
2,370223,276846,276846,19810,724.08,0,724.08,0,0,0


## 2. Check Timestamp Range


In [14]:
df_edges["Timestamp"] = df_edges["Timestamp"] - df_edges["Timestamp"].min()
n_days = int(df_edges["Timestamp"].max() / (3600 * 24) + 1)
n_samples = len(df_edges)
timestamp_range = df_edges["Timestamp"].max()
print(f"Timestamp range: {timestamp_range:,} seconds ({n_days} days, {n_samples:,} transactions)")


Timestamp range : 5,938,140 seconds
Dataset spans   : 69 calendar days
Total samples   : 5,000,000 transactions


## 3. Create Node Feature Matrix (x)


In [15]:
max_n_id = int(df_edges[["from_id", "to_id"]].to_numpy().max()) + 1
x = torch.ones((max_n_id, 1), dtype=torch.float)
print(f"Total unique account nodes: {max_n_id:,}, Node feature matrix x: {x.shape}")


Total unique account nodes (max_n_id): 1,754,264
Node feature matrix shape x: torch.Size([1754264, 1])


## 4. Create Edge Index Tensor (edge_index)


In [16]:
edge_index = torch.LongTensor(df_edges[["from_id", "to_id"]].to_numpy().T)
print(f"Edge index shape: {edge_index.shape}")


edge_index shape: torch.Size([2, 5000000])
Sample edge (from -> to): [920505, 935525]


## 5. Create Edge Feature Tensor (edge_attr)


In [17]:
curr_col = "Received Currency" if "Received Currency" in df_edges.columns else "Receiving Currency"
edge_features = ["Timestamp", "Amount Received", curr_col, "Payment Format"]
edge_attr = torch.tensor(df_edges[edge_features].to_numpy()).float()
print(f"Edge attribute matrix shape: {edge_attr.shape}")


edge_attr shape: torch.Size([5000000, 4])
Edge features included: ['Timestamp', 'Amount Received', 'Received Currency', 'Payment Format']
Sample row 0: [0.0, 2825.699951171875, 10.0, 3.0]


## 6. Create Timestamps and Label Tensor (y)


In [18]:
timestamps = torch.FloatTensor(df_edges["Timestamp"].to_numpy())
y = torch.LongTensor(df_edges["Is Laundering"].to_numpy())
print(f"Labels tensor shape: {y.shape}, Illicit ratio: {y.float().mean()*100:.3f}%")


timestamps shape: torch.Size([5000000])
y label shape: torch.Size([5000000])
Illicit transactions ratio: 0.056%


## 7. Wrap into Data Object


In [20]:
from torch_geometric.data import Data

graph_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
graph_data.timestamps = timestamps
print(f"Graph Data object created with {graph_data.num_nodes:,} nodes and {graph_data.num_edges:,} edges")


Nodes (accounts) : 1,754,264
Edges (transactions) : 5,000,000
Node feature dim (placeholder 1.0s) : 1
Edge feature dim (Timestamp, Amount, Currency, Format) : 4
Timestamps attached : True
Label tensor shape : (5000000,)

Data(x=[1754264, 1], edge_index=[2, 5000000], edge_attr=[5000000, 4], y=[5000000], timestamps=[5000000])


## 8. Save Graph Data Tensors


In [21]:
save_path = "graph_data.pt"
torch.save({"x": x, "edge_index": edge_index, "edge_attr": edge_attr, "timestamps": timestamps, "y": y}, save_path)
print(f"Saved graph_data.pt -> {save_path} ({os.path.getsize(save_path) / 1e9:.2f} GB)")


Saved graph_data.pt -> graph_data.pt
File size: 0.23 GB
